# 99 — View scraped data in Excel

Quick utility notebook to convert the raw `postings_<industry>.csv` files into a single Excel workbook with one sheet per industry plus a combined sheet. Excel/Numbers handles long description text much better than CSV.

**Safe to run while `01_scraping.ipynb` is still scraping.** The scraper writes each CSV via `os.replace()` (atomic at the filesystem level), so any read in this notebook always sees a complete file — never a half-written one. Re-run any time to refresh the Excel view with the latest data.

**Output:** `data/raw/postings_all.xlsx` plus one `postings_<industry>.xlsx` per industry.

In [ ]:
# --- Project-root path bootstrap (added by repo-reorg) ---
# Ensures cwd is the project root regardless of where this notebook was
# launched from (root, code/, JupyterLab tree, VS Code, etc.).
import os
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / 'data').exists() and _here.parent != _here:
    _here = _here.parent
os.chdir(_here)
PROJECT_ROOT = _here
# ---------------------------------------------------------

# Install openpyxl if needed (only run once)
import importlib, subprocess, sys
if importlib.util.find_spec('openpyxl') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'])
print('openpyxl ready')

In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Font, PatternFill

DATA_DIR = Path('data/raw')
OUT_XLSX = DATA_DIR / 'postings_all.xlsx'

# Per-column widths in Excel character units. Description is wide enough to read
# multiple lines without being absurd; row height handles the wrap.
COL_WIDTHS = {
    'job_id':          14,
    'title':           45,
    'company':         30,
    'industry_key':    16,
    'industry_label':  32,
    'seniority':       16,
    'location':        28,
    'date_posted':     14,
    'description':     90,   # wrapped
    'skills_tags':     24,
    'employment_type': 16,
    'source_platform': 14,
    'source_url':      40,
    'scraped_at':      20,
}
DEFAULT_WIDTH = 18
DESCRIPTION_PREVIEW_CHARS = 1500   # truncate for spreadsheet view (full text stays in CSV)
ROW_HEIGHT = 90                    # tall enough to show ~4-5 lines of wrapped text

In [ ]:
import time

def _safe_read_csv(path, retries: int = 5, delay: float = 0.4):
    """Read a CSV with retries.

    Safe to call while the scraper is actively writing: write_postings()
    uses os.replace() so readers always see a complete file. The retry
    here is belt-and-braces in case of any other writer path, transient
    Spotlight indexing locks on macOS, etc.
    """
    last_exc = None
    for attempt in range(retries):
        try:
            return pd.read_csv(path, dtype=str)
        except (pd.errors.ParserError, pd.errors.EmptyDataError, OSError) as exc:
            last_exc = exc
            time.sleep(delay)
    raise last_exc


def load_postings_csvs(data_dir: Path) -> dict:
    """Load every postings_<industry>.csv into a dict keyed by industry.

    Skips any `.tmp` sibling files left over from in-flight atomic writes.
    """
    frames = {}
    for csv_path in sorted(data_dir.glob('postings_*.csv')):
        if csv_path.suffix == '.tmp' or csv_path.name.endswith('.csv.tmp'):
            continue
        industry = csv_path.stem.replace('postings_', '')
        try:
            df = _safe_read_csv(csv_path)
            frames[industry] = df
            print(f'  {industry:20s}  {len(df):>5,} rows  ({csv_path.name})')
        except Exception as exc:
            print(f'  {industry:20s}  FAILED to load: {exc}')
    return frames


print('Loading postings CSVs from', DATA_DIR.resolve())
frames = load_postings_csvs(DATA_DIR)
print(f'\n{len(frames)} industry file(s) loaded.')


In [ ]:
def write_workbook(frames: dict, out_path: Path,
                   preview_chars: int = DESCRIPTION_PREVIEW_CHARS) -> None:
    """
    Write one workbook with:
      - One sheet per industry
      - One 'combined' sheet stacking everything
      - Auto-set column widths
      - Wrap text in description column, fixed row height
      - Bold header row with light gray fill
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)
    combined = pd.concat(frames.values(), ignore_index=True) if frames else pd.DataFrame()

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        # Combined sheet first so it opens by default
        if not combined.empty:
            _write_sheet(writer, combined, 'combined', preview_chars)
        for ind, df in frames.items():
            _write_sheet(writer, df, ind[:31], preview_chars)   # Excel sheet name cap is 31 chars

    print(f'\nWrote {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)')
    print(f'  Combined rows: {len(combined):,}')


def _write_sheet(writer, df: pd.DataFrame, sheet_name: str, preview_chars: int) -> None:
    """Helper: write one DataFrame to a sheet with formatting."""
    # Truncate description for cleaner Excel display (full text remains in CSV)
    df_view = df.copy()
    if 'description' in df_view.columns:
        df_view['description'] = df_view['description'].fillna('').apply(
            lambda t: t if len(t) <= preview_chars else t[:preview_chars] + '...')

    df_view.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]

    header_fill = PatternFill(start_color='DDDDDD', end_color='DDDDDD', fill_type='solid')
    bold        = Font(bold=True)
    wrap_align  = Alignment(wrap_text=True, vertical='top')
    top_align   = Alignment(vertical='top')

    # Header row
    for col_idx, col_name in enumerate(df_view.columns, start=1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = bold
        cell.fill = header_fill
        width = COL_WIDTHS.get(col_name, DEFAULT_WIDTH)
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    # Body rows — alignment + row height
    for row_idx in range(2, ws.max_row + 1):
        ws.row_dimensions[row_idx].height = ROW_HEIGHT
        for col_idx, col_name in enumerate(df_view.columns, start=1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.alignment = wrap_align if col_name == 'description' else top_align

    # Freeze the header row and the first column (job_id) for easier browsing
    ws.freeze_panes = 'B2'

    # Auto-filter on the header so user can sort/filter in Excel
    ws.auto_filter.ref = ws.dimensions

    print(f'  sheet "{sheet_name}" → {len(df_view):,} rows')


write_workbook(frames, OUT_XLSX)

In [ ]:
# Optional: also dump each industry to its own .xlsx file (handy if you only want one in Excel)
for ind, df in frames.items():
    path = DATA_DIR / f'postings_{ind}.xlsx'
    with pd.ExcelWriter(path, engine='openpyxl') as w:
        _write_sheet(w, df, ind[:31], DESCRIPTION_PREVIEW_CHARS)
    print(f'  {path.name}  ({path.stat().st_size / 1024:.1f} KB)')

In [ ]:
# Quick sanity check on description completeness — fooled by Excel display, not by pandas
for ind, df in frames.items():
    if 'description' not in df.columns:
        continue
    desc = df['description'].fillna('')
    print(f'{ind:20s}  rows={len(df):>4}  '
          f'with_desc={(desc.str.len() > 100).sum():>4}  '
          f'mean_chars={int(desc.str.len().mean()):>5}')